# Notebook 03 — Harmonização da Base PEDE

Este notebook prepara `data/pe_de_clean.csv` como uma **base harmonizada** alinhada ao pipeline atual do app e ao `02_modelo_preditivo.ipynb`.

Ele **não** gera mais `app/scaler.pkl`: os artefatos operacionais do modelo permanecem responsabilidade do notebook 02.

In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

INDE_THRESHOLD = 6.5
CSV_NAME = 'BASE DE DADOS PEDE 2024 - DATATHON - PEDE2022.csv'

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data').exists() and (p / 'notebooks').exists()), Path.cwd())
DATA_PATH = ROOT / 'data' / CSV_NAME
OUT_CSV = ROOT / 'data' / 'pe_de_clean.csv'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'CSV bruto não encontrado: {DATA_PATH}')

def slugify_column(name):
    text = unicodedata.normalize('NFKD', str(name).strip().lower())
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'[^a-z0-9]+', '_', text)
    return re.sub(r'_+', '_', text).strip('_')

df_raw = None
for encoding in ['utf-8-sig', 'cp1252', 'latin1']:
    try:
        df_raw = pd.read_csv(DATA_PATH, encoding=encoding)
        print(f'Arquivo lido com encoding: {encoding}')
        break
    except UnicodeDecodeError as exc:
        print(f'Falhou com {encoding}: {exc}')

if df_raw is None:
    raise ValueError('Não foi possível carregar o CSV com os encodings testados.')

df = df_raw.copy()
df.columns = [slugify_column(col) for col in df.columns]
print('ROOT =', ROOT)
print('DATA_PATH =', DATA_PATH)
print('Shape bruto =', df.shape)
print('Amostra de colunas =', df.columns[:12].tolist())

Arquivo lido com encoding: utf-8-sig
ROOT = D:\passos_magicos
DATA_PATH = D:\passos_magicos\data\BASE DE DADOS PEDE 2024 - DATATHON - PEDE2022.csv
Shape bruto = (860, 42)
Amostra de colunas = ['ra', 'fase', 'turma', 'nome', 'ano_nasc', 'idade_22', 'genero', 'ano_ingresso', 'instituicao_de_ensino', 'pedra_20', 'pedra_21', 'pedra_22']


In [2]:
def column_key(value):
    text = unicodedata.normalize('NFKD', str(value).strip().lower())
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return re.sub(r'[^a-z0-9]+', '', text)

def normalize_text(value):
    text = unicodedata.normalize('NFKD', str(value).strip().lower())
    return ''.join(ch for ch in text if not unicodedata.combining(ch))

def to_num(series):
    return pd.to_numeric(series.astype(str).str.replace(',', '.', regex=False), errors='coerce')

def get_matching_col(df_ref, *aliases):
    normalized = {column_key(col): col for col in df_ref.columns}
    for alias in aliases:
        match = normalized.get(column_key(alias))
        if match:
            return match
    return None

def map_pedra(value):
    return {'quartzo': 1, 'agata': 2, 'ametista': 3, 'topazio': 4}.get(normalize_text(value), np.nan)

def dynamic_weights(fase_series):
    fase = pd.to_numeric(fase_series, errors='coerce').fillna(0)
    fase_8 = fase >= 8
    return {
        'ian_num': np.where(fase_8, 0.00, 0.10),
        'ida_num': np.where(fase_8, 0.40, 0.20),
        'ieg_num': np.full(len(fase), 0.20),
        'iaa_num': np.where(fase_8, 0.00, 0.10),
        'ips_num': np.full(len(fase), 0.20),
        'ipp_num': np.full(len(fase), 0.10),
        'ipv_num': np.full(len(fase), 0.10),
    }

def reverse_rank_score(series):
    numeric = to_num(series)
    valid = numeric.dropna()
    if valid.empty or valid.max() == valid.min():
        return pd.Series(np.nan, index=series.index, dtype=float), {'min': np.nan, 'max': np.nan}
    lower = float(valid.min())
    upper = float(valid.max())
    score = 10 * (1 - (numeric - lower) / (upper - lower))
    return score.clip(0, 10).astype(float), {'min': lower, 'max': upper}

PALAVRAS_NEG = {
    'melhorar', 'empenhar', 'dificuldade', 'atraso', 'deficit', 'atencao',
    'problema', 'risco', 'alerta', 'comportamento', 'limitacao', 'preocupa',
    'atenção', 'limitação', 'preocupação'
}
PALAVRAS_POS = {
    'destaque', 'excelente', 'promovido', 'bolsa', 'lider', 'potencial',
    'engajado', 'comprometido', 'aprovado', 'evolucao', 'crescimento',
    'líder', 'evolução'
}

def score_sentimento(texto):
    if pd.isna(texto):
        return 0
    content = str(texto).lower()
    return sum(1 for word in PALAVRAS_POS if word in content) - sum(1 for word in PALAVRAS_NEG if word in content)

print('✅ Funções auxiliares carregadas.')

✅ Funções auxiliares carregadas.


In [3]:
def set_numeric(df_ref, target, *aliases):
    source = get_matching_col(df_ref, target, *aliases)
    if source is not None:
        df_ref[target] = to_num(df_ref[source])
    return source

def build_clean_base(df_ref):
    df_ref = df_ref.copy()

    for col in ['fase', 'ano_nasc', 'idade_22', 'ano_ingresso', 'inde_22', 'cg', 'cf', 'ct', 'iaa', 'ieg', 'ips', 'ida', 'matem', 'portug', 'ingles', 'ipv', 'ian', 'defas']:
        set_numeric(df_ref, col)
    set_numeric(df_ref, 'no_av', 'n_av', 'Nº Av', 'N° Av', 'N Av')

    required = {
        'fase': get_matching_col(df_ref, 'fase'),
        'inde_22': get_matching_col(df_ref, 'inde_22'),
        'ian': get_matching_col(df_ref, 'ian'),
        'ida': get_matching_col(df_ref, 'ida'),
        'ieg': get_matching_col(df_ref, 'ieg'),
        'iaa': get_matching_col(df_ref, 'iaa'),
        'ips': get_matching_col(df_ref, 'ips'),
        'ipv': get_matching_col(df_ref, 'ipv'),
        'cf': get_matching_col(df_ref, 'cf'),
        'ct': get_matching_col(df_ref, 'ct'),
        'pedra_22': get_matching_col(df_ref, 'pedra_22'),
        'pedra_21': get_matching_col(df_ref, 'pedra_21'),
    }
    missing = [name for name, col in required.items() if col is None]
    if missing:
        raise ValueError(f'Colunas obrigatórias ausentes: {missing}')

    df_ref['inde_num'] = df_ref['inde_22']
    df_ref['inde_22_num'] = df_ref['inde_22']
    df_ref['cg_num'] = df_ref['cg'] if 'cg' in df_ref.columns else np.nan
    df_ref['ian_num'] = df_ref['ian']
    df_ref['ida_num'] = df_ref['ida']
    df_ref['ieg_num'] = df_ref['ieg']
    df_ref['iaa_num'] = df_ref['iaa']
    df_ref['ips_num'] = df_ref['ips']
    df_ref['ipv_num'] = df_ref['ipv']
    df_ref['matem_num'] = df_ref['matem'] if 'matem' in df_ref.columns else np.nan
    df_ref['portug_num'] = df_ref['portug'] if 'portug' in df_ref.columns else np.nan
    df_ref['ingles_num'] = df_ref['ingles'] if 'ingles' in df_ref.columns else np.nan

    df_ref['pedra_22_raw'] = df_ref[required['pedra_22']].astype(str).str.strip()
    df_ref['pedra_21_raw'] = df_ref[required['pedra_21']].astype(str).str.strip()
    df_ref['pedra_22_num'] = df_ref['pedra_22_raw'].apply(map_pedra)
    df_ref['pedra_21_num'] = df_ref['pedra_21_raw'].apply(map_pedra)
    df_ref['evolucao_pedra'] = df_ref['pedra_22_num'] - df_ref['pedra_21_num']

    cf_score, cf_bounds = reverse_rank_score(df_ref[required['cf']])
    ct_score, ct_bounds = reverse_rank_score(df_ref[required['ct']])
    df_ref['ipp_num'] = pd.concat([cf_score, ct_score], axis=1).mean(axis=1)
    df_ref['ipp'] = df_ref['ipp_num']

    text_cols = [col for col in df_ref.columns if any(tag in column_key(col) for tag in ['recav', 'recpsicologia', 'destaque'])]
    if text_cols:
        sent_parts = [df_ref[col].apply(score_sentimento) for col in text_cols]
        len_parts = [df_ref[col].apply(lambda x: len(str(x)) if not pd.isna(x) else 0) for col in text_cols]
        df_ref['sent_score'] = pd.concat(sent_parts, axis=1).sum(axis=1)
        df_ref['sent_len'] = pd.concat(len_parts, axis=1).sum(axis=1)
    else:
        df_ref['sent_score'] = 0.0
        df_ref['sent_len'] = 0.0

    weights = dynamic_weights(df_ref['fase'])
    acad_den = weights['ian_num'] + weights['ida_num']
    psic_den = weights['ieg_num'] + weights['iaa_num'] + weights['ips_num']
    peda_den = weights['ipp_num'] + weights['ipv_num']

    df_ref['dim_academica'] = (df_ref['ian_num'] * weights['ian_num'] + df_ref['ida_num'] * weights['ida_num']) / acad_den
    df_ref['dim_psicossocial'] = (df_ref['ieg_num'] * weights['ieg_num'] + df_ref['iaa_num'] * weights['iaa_num'] + df_ref['ips_num'] * weights['ips_num']) / psic_den
    df_ref['dim_psicopedagogica'] = (df_ref['ipp_num'] * weights['ipp_num'] + df_ref['ipv_num'] * weights['ipv_num']) / peda_den
    df_ref['inde_calc'] = (
        df_ref['ian_num'] * weights['ian_num']
        + df_ref['ida_num'] * weights['ida_num']
        + df_ref['ieg_num'] * weights['ieg_num']
        + df_ref['iaa_num'] * weights['iaa_num']
        + df_ref['ips_num'] * weights['ips_num']
        + df_ref['ipp_num'] * weights['ipp_num']
        + df_ref['ipv_num'] * weights['ipv_num']
    )
    df_ref['em_risco'] = (df_ref['inde_num'] < INDE_THRESHOLD).astype(int)

    front_cols = [
        'ra', 'fase', 'turma', 'nome', 'ano_nasc', 'idade_22', 'genero', 'ano_ingresso',
        'instituicao_de_ensino', 'pedra_20', 'pedra_21', 'pedra_22', 'pedra_21_raw', 'pedra_22_raw',
        'pedra_21_num', 'pedra_22_num', 'evolucao_pedra', 'inde_22', 'inde_22_num', 'inde_num',
        'cg', 'cf', 'ct', 'no_av', 'iaa', 'ieg', 'ips', 'ida', 'matem', 'portug', 'ingles', 'ipv', 'ian',
        'ipp', 'ipp_num', 'cg_num', 'iaa_num', 'ieg_num', 'ips_num', 'ida_num', 'matem_num', 'portug_num',
        'ingles_num', 'ipv_num', 'ian_num', 'dim_academica', 'dim_psicossocial', 'dim_psicopedagogica',
        'sent_score', 'sent_len', 'inde_calc', 'defas', 'em_risco'
    ]
    ordered = [col for col in front_cols if col in df_ref.columns]
    remaining = [col for col in df_ref.columns if col not in ordered]
    df_ref = df_ref[ordered + remaining]

    ipp_bounds = {'cf': cf_bounds, 'ct': ct_bounds, 'source': 'reverse_rank_cf_ct'}
    return df_ref, ipp_bounds, text_cols

df_clean, ipp_bounds, cols_rec = build_clean_base(df)
preview_cols = ['ra', 'fase', 'inde_num', 'ipp_num', 'dim_academica', 'dim_psicossocial', 'dim_psicopedagogica', 'inde_calc', 'em_risco']
print(df_clean[[col for col in preview_cols if col in df_clean.columns]].head())

     ra  fase  inde_num   ipp_num  dim_academica  dim_psicossocial  \
0  RA-1     7     5.783  6.907915       4.333333              5.54   
1  RA-2     7     7.055  9.228519       7.866667              6.36   
2  RA-3     7     6.591  8.215276       7.066667              5.40   
3  RA-4     7     5.951  7.868802       6.666667              5.80   
4  RA-5     7     7.427  9.574992       6.800000              7.26   

   dim_psicopedagogica  inde_calc  em_risco  
0             7.092957   5.488591         1  
1             8.003259   7.140652         0  
2             7.885638   6.397128         0  
3             6.573401   6.214680         1  
4             8.481996   7.366399         0  


In [4]:
summary_cols = ['fase', 'inde_num', 'ian_num', 'ida_num', 'ieg_num', 'iaa_num', 'ips_num', 'ipp_num', 'ipv_num', 'dim_academica', 'dim_psicossocial', 'dim_psicopedagogica', 'inde_calc', 'sent_score', 'sent_len', 'em_risco']
missing_pct = df_clean[summary_cols].isna().mean().mul(100).round(1).sort_values(ascending=False)

print('Campos textuais usados no NLP:', cols_rec if cols_rec else 'nenhum')
print('Bounds da proxy de IPP:', ipp_bounds)
print('Distribuição de em_risco = (inde_num < 6.5):')
print(df_clean['em_risco'].value_counts(normalize=True).rename({0: 'Sem risco', 1: 'Em risco'}).round(3))
print('Missing (%), apenas campos com ausência:')
print(missing_pct[missing_pct > 0])

df_clean.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print('Base clean harmonizada salva em:', OUT_CSV)
print('Shape final:', df_clean.shape)

Campos textuais usados no NLP: ['rec_av1', 'rec_av2', 'rec_av3', 'rec_av4', 'rec_psicologia', 'destaque_ieg', 'destaque_ida', 'destaque_ipv']
Bounds da proxy de IPP: {'cf': {'min': 1.0, 'max': 192.0}, 'ct': {'min': 1.0, 'max': 18.0}, 'source': 'reverse_rank_cf_ct'}
Distribuição de em_risco = (inde_num < 6.5):
em_risco
Sem risco    0.748
Em risco     0.252
Name: proportion, dtype: float64
Missing (%), apenas campos com ausência:
Series([], dtype: float64)
Base clean harmonizada salva em: D:\passos_magicos\data\pe_de_clean.csv
Shape final: (860, 68)


## Observações finais

- `data/pe_de_clean.csv` passa a registrar uma base harmonizada com colunas brutas relevantes e features derivadas do pipeline atual.
- `IPP`, dimensões consolidadas, `inde_calc` e `em_risco` seguem a mesma lógica-base do app e do notebook 02.
- O treinamento do modelo e a serialização de artefatos continuam no `02_modelo_preditivo.ipynb`.